In [ ]:
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = [ "pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score" ]

reward_model_df_dict = {}
first_image_names = None
first_prompts = None
for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    reward_model_df_dict[reward_model] = reward_model_df

    if first_image_names is None:
        first_image_names = reward_model_df["image_name"].tolist()
        first_prompts = reward_model_df["prompt"].tolist()
    else:
        if reward_model_df["image_name"].tolist() != first_image_names:
            print(f"Warning: Inconsistent image names or order found in {reward_model}.")
        
        if reward_model_df["prompt"].tolist() != first_prompts:
            print(f"Warning: Inconsistent prompts found in {reward_model}.")
            
scores = []
for reward_model in reward_model_list:
    scores.append(reward_model_df_dict[reward_model][reward_model])
scores_df = pd.DataFrame(scores).T  # -> reward_models as columns
print(f"len(scores_df)={len(scores_df)}")
corr_matrix, _ = spearmanr(scores_df, axis=0)
plt.figure(figsize=(10, 8), dpi=300)
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", xticklabels=reward_model_list, yticklabels=reward_model_list, vmin=-1, vmax=1, fmt=".2f")
plt.title('Spearman Rank Correlation between Reward Models')
plt.show()

In [ ]:
# aggregate reward_model scores
import os
import pandas as pd

base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_cfg_1.0/drawbench-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = [ "pickscore", "hpsv2", "imagereward" ]
target_reward_model = "-".join(reward_model_list)
reward_model_df_dict = {}

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    reward_model_df_dict[reward_model] = reward_model_df

target_score = pd.Series([0] * len(reward_model_df_dict[reward_model_list[0]]))
for reward_model, reward_model_df in reward_model_df_dict.items():
    target_score += reward_model_df[reward_model]
    
target_df = reward_model_df_dict[reward_model_list[0]].copy()
target_df[target_reward_model] = target_score

output_path = os.path.join(base_reward_score_dir, f"{target_reward_model}.csv")
target_df.to_csv(output_path, index=False)


In [ ]:
### Grouped correlation analysis ###
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code"]
# reward_model_list = ["clip_iqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    # print(f"reward_model: {reward_model}")
    # print(f"Columns for {reward_model_df[reward_model].iloc[0]}:", reward_model_df.columns)
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

grouped_by_prompt = total_df.groupby('prompt')
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)
axes = axes.flatten()
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    if prompt == "an overgrown abandoned red barn, covered in vines, sunlight filtering through, a stag deer standing in the entrance, 4k": 
        print(f"idx: {idx}")
        continue

    axes_idx = idx if idx < 3 else idx-1
    scores = group[reward_model_list].values 
    corr_matrix, _ = spearmanr(scores, axis=0)
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", xticklabels=reward_model_list, yticklabels=reward_model_list, vmin=-1, vmax=1, ax=axes[axes_idx], fmt=".2f")
    axes[axes_idx].set_title(f'Spearman Rank Correlation for Prompt:\n{prompt}')
    
plt.tight_layout()
plt.show()


In [ ]:
### show image  ###
import os
import pandas as pd
from scipy.stats import spearmanr
import seaborn as sns
from PIL import Image
import matplotlib.pyplot as plt

base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/images"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
# reward_model_list = [ "pickscore", "hpsv2", "imagereward", "code" ]
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clip_iqa", "deqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []
font_size = 6
for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])

total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

grouped_by_prompt = total_df.groupby('prompt')
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    if idx != 2: continue
    num_top_bottom, num_reward_models = 5, len(reward_model_list)
    num_cols, num_rows = num_top_bottom * 2, len(reward_model_list)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 3, num_rows * 3.5), dpi=200)
    
    for col_idx in range(num_cols):
        for row_idx, reward_model in enumerate(reward_model_list):
            ax = axes[row_idx][col_idx]
            sorted_df = group.sort_values(by=reward_model, ascending=False, ignore_index=True)
            
            if col_idx < num_top_bottom:
                row_data = sorted_df.loc[col_idx]
                if row_idx == 0:
                    ax.set_title(f"Rank #{col_idx + 1}", fontsize=9)
            else:
                bottom_idx = col_idx - num_top_bottom
                row_data = sorted_df.tail(num_top_bottom).iloc[bottom_idx]
                if row_idx == 0:
                    total_images = len(group)
                    rank = total_images - (num_top_bottom - 1) + bottom_idx
                    ax.set_title(f"Rank #{rank}", fontsize=font_size, color='red')
            
            image_name = row_data['image_name']
            score = row_data[reward_model]
            image_path = os.path.join(base_image_dir, f"{image_name}.png")
            ax.imshow(Image.open(image_path))
            ax.set_xlabel(f"image_name: {image_name}, score: {score:.2f}", fontsize=font_size)

            if col_idx == 0: ax.set_ylabel(reward_model, fontsize=font_size)
            ax.set_xticks([])
            ax.set_yticks([])
    
    fig.suptitle(f'Prompt: {prompt}', fontsize=12, y=1.005)
    plt.tight_layout()
    plt.show()

In [ ]:
### hard example analysis - (两级分化 score) - get example ###
import os
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

reward_model = "b_free"
base_reward_model_score_dir = f"/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score/{reward_model}.csv"
reward_model_df = pd.read_csv(base_reward_model_score_dir)
grouped_by_prompt = reward_model_df.groupby('prompt')

# hard_prompts = []
# for idx, (prompt, group) in enumerate(grouped_by_prompt):
#     low = (group[reward_model] < 0.05).sum()
#     mid = ((group[reward_model] >= 0.05) & (group[reward_model] <= 0.3+0.05)).sum()
#     high = (group[reward_model] > 0.05+0.3).sum()
#     if low >= 12 and high >=3: hard_prompts.append((prompt, low, mid, high))
    
# for p, low, mid, high in hard_prompts:
#     print(f"{p} - low:{low}, high: {high}")
hard_prompts = []
for prompt, group in reward_model_df.groupby('prompt'):
    scores = group[reward_model].values.reshape(-1, 1)
    if len(scores) != 24:
        continue
    
    # low = (group[reward_model] < 0.02).sum()
    # if low <= 6: continue
    kmeans = KMeans(n_clusters=2, random_state=0).fit(scores)
    labels = kmeans.labels_
    
    unique, counts = np.unique(labels, return_counts=True)
    if min(counts) < 3:
        continue  
    
    sil_score = silhouette_score(scores, labels)
    if sil_score > 0.75:
        hard_prompts.append((prompt, sil_score))
hard_prompts = sorted(hard_prompts, key=lambda x: x[1], reverse=True)
print(f"len hard_prompts: {len(hard_prompts)}")
for p, score in hard_prompts:
    print(f"{p} - silhouette_score: {score:.4f}")

In [ ]:
#### print specific prompt score distribution (min, max, mean, std) ####
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

analysis_reward_model = "code-resize_256_crop_224"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code-resize_256_crop_224", "code-crop_224", "b_free", "dinov2"]
# reward_model_list = ["clip_iqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

hard_prompts = []
grouped_by_prompt = total_df.groupby('prompt')
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    scores = group[analysis_reward_model].values
    mean = scores.mean()
    max = scores.max()
    min = scores.min()
    low = (scores < 0.1).sum()
    high = (scores > 0.4).sum()
    if low >=3 and high >= 3:
        print(f"mean: {mean}, max: {max}, min: {min}, prompt: {prompt}")
        hard_prompts.append(prompt) #
    # if mean > 0.5:
    #     hard_prompts.append(prompt)
print(f"len hard_prompts: {len(hard_prompts)}")

In [ ]:
### hard example analysis - (两级分化 score) - visualization correlation matrix ###
import os
import math
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# hard_prompts = [
#     "a two headed slug at a fork in the road",
#     "beautiful asian girl with huge naturals in pool",
#     "britney spears in 6 in stripper heels",
#     "dissected selfieharm pregnant girl at hospital"
# ]

# --- 配置路径和模型 ---
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = [
    "pickscore", "hpsv2", "imagereward", "clipscore", "vqascore",
    "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", analysis_reward_model
]

# --- 加载并合并所有 reward 数据 ---
all_data = []
for reward_model in reward_model_list:
    p = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    df = pd.read_csv(p).sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(df[['prompt', 'image_name', reward_model]])

total_df = all_data[0]
for i in range(1, len(all_data)):
    rm_name = reward_model_list[i]
    total_df = pd.merge(total_df, all_data[i], on=['prompt', 'image_name'], how='outer')


print(f"hard_prompts count: {len(hard_prompts)}")

# --- 准备绘图 ---
# hard_prompts = [p for p, score in hard_prompts]
# hard_prompts = [
#     "a happy nepalese girl in a village",
#     "genere un perfil de rostro de 3/4 de una mujer sofisticada de 23 años latina, alegre, cabello negro",
#     "General Peron riding an Atomic Bomb",
#     "A digital illustration of a dragon flying over a castle, fantasy style"
# ]
grouped_by_prompt = total_df.groupby('prompt')
hard_prompts_sorted = sorted(list(hard_prompts))
n = len(hard_prompts_sorted)

if n == 0:
    print("No hard prompts to plot.")
else:
    # 动态设置子图网格：尽量接近正方形，或 2 行多列
    cols = min(n, 2)  # 最多每行 4 个，避免太宽
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows), dpi=200)
    
    # 如果只有一个 prompt，axes 不是数组，需统一处理
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, prompt in enumerate(hard_prompts_sorted):
        ax = axes[i]
        try:
            group = grouped_by_prompt.get_group(prompt).copy()
        except KeyError:
            print(f"Prompt not found in data: {prompt}")
            ax.axis('off')
            continue

        # 只保留 reward_model_list 中无缺失的行
        sub = group.dropna(subset=reward_model_list)
        if len(sub) < 2:
            print(f"Not enough valid scores for prompt: {prompt}")
            ax.axis('off')
            continue

        scores = sub[reward_model_list].values
        corr_matrix, _ = spearmanr(scores, axis=0)

        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            xticklabels=reward_model_list,
            yticklabels=reward_model_list,
            ax=ax,
            cbar_kws={"shrink": 0.7}
        )
        ax.set_title(f'Prompt:\n{prompt[:60]}...', fontsize=10)  # 截断过长 prompt

    # 隐藏多余的子图
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
### hard example analysis - (dirrectly get high correlation example) ###
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

analysis_reward_model = "code"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code", "b_free"]
# reward_model_list = ["clip_iqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    # print(f"reward_model: {reward_model}")
    # print(f"Columns for {reward_model_df[reward_model].iloc[0]}:", reward_model_df.columns)
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

hard_prompts = []
target_models = ["clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score"]
grouped_by_prompt = total_df.groupby('prompt')
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    scores = group[reward_model_list].values 
    corr_matrix, _ = spearmanr(scores, axis=0)
    corr_df = pd.DataFrame(corr_matrix, index=reward_model_list, columns=reward_model_list)
    valid = sum(corr_df.loc[analysis_reward_model, m] > 0.4 for m in target_models) >= 3
    if valid: hard_prompts.append(prompt)

print(f"len hard_prompts: {len(hard_prompts)}")
print(hard_prompts)

In [ ]:
### hard example analysis - (dirrectly get high correlation example) - visualization correlation matrix ###
import os
import math
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# hard_prompts = []

base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = [
    "pickscore", "hpsv2", "imagereward", "clipscore", "vqascore",
    "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code"
]

all_data = []
for reward_model in reward_model_list:
    p = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    df = pd.read_csv(p).sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(df[['prompt', 'image_name', reward_model]])

total_df = all_data[0]
for i in range(1, len(all_data)):
    rm_name = reward_model_list[i]
    total_df = pd.merge(total_df, all_data[i], on=['prompt', 'image_name'], how='outer')


print(f"hard_prompts count: {len(hard_prompts)}")

grouped_by_prompt = total_df.groupby('prompt')
hard_prompts_sorted = sorted(list(hard_prompts))
n = len(hard_prompts_sorted)

if n == 0:
    print("No hard prompts to plot.")
else:
    # 动态设置子图网格：尽量接近正方形，或 2 行多列
    cols = min(n, 2)  # 最多每行 4 个，避免太宽
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows), dpi=200)
    
    # 如果只有一个 prompt，axes 不是数组，需统一处理
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, prompt in enumerate(hard_prompts_sorted):
        ax = axes[i]
        try:
            group = grouped_by_prompt.get_group(prompt).copy()
        except KeyError:
            print(f"Prompt not found in data: {prompt}")
            ax.axis('off')
            continue

        # 只保留 reward_model_list 中无缺失的行
        sub = group.dropna(subset=reward_model_list)
        if len(sub) < 2:
            print(f"Not enough valid scores for prompt: {prompt}")
            ax.axis('off')
            continue

        scores = sub[reward_model_list].values
        corr_matrix, _ = spearmanr(scores, axis=0)

        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            xticklabels=reward_model_list,
            yticklabels=reward_model_list,
            ax=ax,
            cbar_kws={"shrink": 0.7}
        )
        ax.set_title(f'Prompt:\n{prompt[:60]}...', fontsize=10)  # 截断过长 prompt

    # 隐藏多余的子图
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
### hard example analysis - (dirrectly get high correlation example) ###
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

analysis_reward_model = "code"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code", "b_free"]
# reward_model_list = ["clip_iqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

hard_prompts = []
target_models = ["clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score"]
grouped_by_prompt = total_df.groupby('prompt')
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    code_mean = group['code'].mean()
    if code_mean < 0.3: hard_prompts.append(prompt)
print(f"len hard_prompts: {len(hard_prompts)}")
print(hard_prompts)


In [1]:
#### print specific prompt score distribution (min, max, mean, std) ####
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

analysis_reward_model = "code-resize_256_crop_224"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code-resize_256_crop_224", "b_free"]
# reward_model_list = ["clip_iqa", "aesthetic_v2_5", "code"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

hard_prompts = []
grouped_by_prompt = total_df.groupby('prompt')
for idx, (prompt, group) in enumerate(grouped_by_prompt):
    scores = group[analysis_reward_model].values
    
    # low_count = (scores < 0.1).sum()
    # high_count = (scores > 0.4).sum()
    # if low_count >=4 and high_count >= 4:
    #     hard_prompts.append(prompt)
    
    
    max = scores.max()
    min = scores.min()
    if max - min > 0.4:
        # width = (max - min) / 4.
        # q1 = min + width
        # q2 = min + 2*width
        # q3 = min + 3*width
        
        # count_first = int(np.sum((scores >= min) & (scores < q1)))
        # count_last  = int(np.sum((scores >= q3)   & (scores <= max)))
        
        # if count_first >= 2 and count_last >= 2:
        #     hard_prompts.append(prompt)
        hard_prompts.append(prompt)
    
    # max_score, min_score = scores.max(), scores.min()
    # if max_score - min_score > 0.2:
    #     hard_prompts.append(prompt)

print(f"len hard_prompts: {len(hard_prompts)}")
for p in hard_prompts:
    print(p)

len hard_prompts: 85
"A baby connected to a neural network in a bunker 😂"
3d render of an obsidian-clad dark knight, with a menacing aura, and a faint red glow from his eyes barely seen from under his helmet, scratches and imperfections, ornate detail, warrior pose , accurate color grading, rvb, Sony A7 III, Sigma Art 85mm DG HSM, perfect, subsurface scattering, unreal engine 5
A black Rose with logo of Bitcoin on a propaganda poster alone in the dessert
A blue mill in a sunset
A digital illustration of a dragon flying over a castle, fantasy style
A giraffe by escher, insanely detailed, photorealistic, 8k, ultra high resolution, volumetric lighting, taken with canon eos,
A hi res photo of a beautiful young woman named curvy Jo, wearing a tight sweater
A pencil illustration of a fanged grim reaper
A photo of a beautiful young woman, cute
A photo of a person with the head of a cow, wearing a tuxedo and black bowtie. Beach wallpaper in the background
A photo of a statue of a dog made from

In [ ]:
### hard example analysis - (lpips, SSIM) ###
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import lpips
import torch
from PIL import Image
from itertools import combinations
import numpy as np
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = lpips.LPIPS(net='vgg').to(device) # VGG: Correct; Alex: Fast

analysis_reward_model = "code"
base_reward_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/reward_score"
reward_model_list = ["pickscore", "hpsv2", "imagereward", "clipscore", "vqascore", "clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score", "code", "b_free"]

reward_model_df_dict = {}
first_image_names = None
all_data = []

for reward_model in reward_model_list:
    reward_model_path = os.path.join(base_reward_score_dir, f"{reward_model}.csv")
    reward_model_df = pd.read_csv(reward_model_path)
    reward_model_df = reward_model_df.sort_values(by="image_name", ascending=True, ignore_index=True)
    all_data.append(reward_model_df[['prompt', 'image_name', reward_model]])
    
total_df = all_data[0]
for reward_model_idx, reward_model in enumerate(reward_model_list):
    if reward_model_idx == 0: continue
    total_df = pd.merge(total_df, all_data[reward_model_idx][['prompt', 'image_name', reward_model]], on=['prompt', 'image_name'], how='outer')

image_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusionnft/generate_images/sd3_textencoder_3_none_cfg_1.0/pickscore-analysis/SD3.5M-DiffusionNFT-MultiReward/ckpt-0/images"

def load_image(image_path):
    img = lpips.im2tensor(lpips.load_image(image_path))
    return img.to(device)

def compute_group_lpips(image_paths):
    images = []
    for path in image_paths:
        try:
            images.append(load_image(path))
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None  
    
    if len(images) != 24:
        print(f"Warning: Group has {len(images)} images, not 24.")
        return None
    
    lpips_scores = []
    for img1, img2 in combinations(images, 2):
        with torch.no_grad():
            lpips_scores.append(loss_fn(img1, img2).item())
    
    avg_lpips = np.mean(lpips_scores)
    return avg_lpips

def compute_group_ssim(image_paths):
    images_np = []
    for path in image_paths:
        img = np.array(Image.open(path).convert('RGB'))
        images_np.append(img)
    
    ssim_scores = []
    for img1, img2 in combinations(images_np, 2):
        ssim_scores.append(ssim(img1, img2, channel_axis=-1, data_range=255))
    
    avg_ssim = np.mean(ssim_scores)
    return 1 - avg_ssim

lpips_list = []
ssim_list = []
hard_prompts = []
lpips_threshold = 0.3
target_models = ["clip_iqa", "deqa", "aesthetic", "aesthetic_v2_5", "vila_score"]
grouped_by_prompt = total_df.groupby('prompt')

for idx, (prompt, group) in enumerate(tqdm(grouped_by_prompt, total=len(grouped_by_prompt))):
    image_names = group['image_name'].tolist()
    image_paths = [os.path.join(image_dir, f"{img_name}.png") for img_name in image_names]
    
    avg_lpips = compute_group_lpips(image_paths)
    if avg_lpips is None:
        continue 
    
    # print(f"Prompt {idx}: {prompt[:50]}... | Avg LPIPS: {avg_lpips:.4f}")
    
    # 可选：加 SSIM
    avg_ssim_diff = compute_group_ssim(image_paths)
    print(f" | Avg SSIM diff: {avg_ssim_diff:.4f}")

    # lpips_list.append(avg_lpips)
    ssim_list.append(avg_ssim_diff)    

# 可选：可视化所有 prompt 的 LPIPS 分布
sns.histplot(ssim_list)
plt.show()